# Multi-Class Classification with Undersampling and Scaling
This notebook evaluates multiple machine learning models for SDN traffic classification using an **Undersampling** strategy to address class imbalance.

### Methodology:
1.  **Data Loading**: Load preprocessed traffic data.
2.  **Split-then-Balance**: To prevent data leakage, we perform a stratified train-test split *first*, and then undersample the **training set only**.
3.  **Undersampling Strategy**: Downsample the majority classes in the training set to match a target count, ensuring the model is not biased towards frequent traffic types while keeping the test set representative of real-world distributions.
4.  **Scaling**: Apply model-specific scaling (Standard or Min-Max) after balancing.
5.  **Multi-Split Evaluation**: Test across 70-30, 60-40, 50-50, and 30-70 splits.


In [1]:
import pandas as pd
import numpy as np
import time
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix, 
                             roc_auc_score, cohen_kappa_score)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC 
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import StackingClassifier
from scipy.special import softmax

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("preprocessed_data.csv")
print("Dataset Shape:", df.shape)

X = df.drop("Category", axis=1)
y_multi = df["Category"]

print("Categories discovered:", y_multi.unique())
print("Class Distribution:\n", y_multi.value_counts())

Dataset Shape: (221564, 37)
Categories discovered: [5 3 6 0 2 1 4]
Class Distribution:
 Category
2    164974
0     53551
6      1487
5      1133
3       305
1        91
4        23
Name: count, dtype: int64


In [3]:
# Hybrid Model Definitions
hybrid_dt_svm = StackingClassifier(
    estimators=[("dt", DecisionTreeClassifier(max_depth=20, random_state=42))],
    final_estimator=LinearSVC(dual=False, max_iter=2000),
    cv=2, n_jobs=-1
)

hybrid_knn_svm = StackingClassifier(
    estimators=[("knn", KNeighborsClassifier(n_neighbors=5, n_jobs=-1))],
    final_estimator=LinearSVC(dual=False, max_iter=2000),
    cv=2, n_jobs=-1
)

# Model Map
models = {
    "SVM": LinearSVC(dual=False, random_state=42, max_iter=10000),
    "DT": DecisionTreeClassifier(random_state=42, max_depth=20),
    "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "XGBoost": XGBClassifier(eval_metric="mlogloss", random_state=42, tree_method="hist"),
    "Hybrid DT-SVM": hybrid_dt_svm,
    "Hybrid KNN-SVM": hybrid_knn_svm
}

# Scaler Mapping (Based on Paper Methodology)
scalers = {
    "SVM": MinMaxScaler(),
    "DT": StandardScaler(),
    "KNN": MinMaxScaler(),
    "XGBoost": MinMaxScaler(),
    "Hybrid DT-SVM": MinMaxScaler(),
    "Hybrid KNN-SVM": MinMaxScaler()
}

## Training and Evaluation with Split-First Undersampling
In the loop below, we iterate through different train-test splits. For each split:
1. We split the raw data.
2. We identify the size of the smallest class in the training set (or a fixed minimum) and undersample all other classes to that size.
3. This ensures the model learns features from all classes equally without "seeing" the test data distribution during the balancing phase.


In [4]:
splits = {"70-30": 0.30, "60-40": 0.40, "50-50": 0.50, "30-70": 0.70}
results_all = []

for split_name, test_size in splits.items():
    print(f"\n{'='*60}\nRun: {split_name} | Undersampling & Training\n{'='*60}", flush=True)
    
    # 1. Stratified Split (BEFORE balancing)
    X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
        X, y_multi, test_size=test_size, random_state=42, stratify=y_multi
    )
    
    # 2. Undersample the training set ONLY
    # Find the count of the smallest class to balance towards it
    # Note: We can also set a fixed cap (e.g., 5000) if classes are extremely large
    min_class_size = y_train_raw.value_counts().min()
    print(f"  Balancing classes to size: {min_class_size}...", end="", flush=True)
    
    train_df = X_train_raw.copy()
    train_df['Category'] = y_train_raw
    
    under_sampled_dfs = []
    for label in train_df['Category'].unique():
        class_subset = train_df[train_df['Category'] == label]
        under_sampled_dfs.append(class_subset.sample(n=min_class_size, random_state=42))
        
    balanced_train_df = pd.concat(under_sampled_dfs).sample(frac=1, random_state=42)
    X_train_res = balanced_train_df.drop('Category', axis=1)
    y_train_res = balanced_train_df['Category']
    print(f" Done. (Resampled size: {len(X_train_res)})", flush=True)

    for name, model in models.items():
        print(f"  Training {name}... ", end="", flush=True)
        start = time.time()
        
        # 3. Scale (Fit on Balanced Train, Transform Test)
        scaler = scalers.get(name, MinMaxScaler())
        Xt_scaled = scaler.fit_transform(X_train_res)
        Xv_scaled = scaler.transform(X_test_raw)
        
        # 4. Train
        model.fit(Xt_scaled, y_train_res)
        t_diff = time.time() - start
        
        # 5. Predict and Metrics
        y_pred = model.predict(Xv_scaled)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        kappa = cohen_kappa_score(y_test, y_pred)
        
        # ROC AUC
        try:
            if hasattr(model, "predict_proba"):
                y_prob = model.predict_proba(Xv_scaled)
            elif hasattr(model, "decision_function"):
                y_prob = softmax(model.decision_function(Xv_scaled), axis=1)
            else:
                y_prob = None
            roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovr') if y_prob is not None else 0.0
        except:
            roc_auc = 0.0
            
        results_all.append({
            "Split": split_name, "Model": name, "Accuracy": acc, 
            "F1 Score": f1, "Kappa Score": kappa, "ROC AUC": roc_auc, "Time": t_diff
        })
        print(f"Done ({t_diff:.2f}s) | Acc: {acc:.4f}", flush=True)


Run: 70-30 | Undersampling & Training
  Balancing classes to size: 16... Done. (Resampled size: 112)
  Training SVM... Done (0.04s) | Acc: 0.4786
  Training DT... Done (0.04s) | Acc: 0.7795
  Training KNN... Done (0.03s) | Acc: 0.5125
  Training XGBoost... Done (0.16s) | Acc: 0.9248
  Training Hybrid DT-SVM... Done (4.67s) | Acc: 0.7795
  Training Hybrid KNN-SVM... Done (3.81s) | Acc: 0.5401

Run: 60-40 | Undersampling & Training
  Balancing classes to size: 14... Done. (Resampled size: 98)
  Training SVM... Done (0.05s) | Acc: 0.6700
  Training DT... Done (0.05s) | Acc: 0.7390
  Training KNN... Done (0.06s) | Acc: 0.5372
  Training XGBoost... Done (0.18s) | Acc: 0.7894
  Training Hybrid DT-SVM... Done (3.60s) | Acc: 0.7390
  Training Hybrid KNN-SVM... Done (3.77s) | Acc: 0.5589

Run: 50-50 | Undersampling & Training
  Balancing classes to size: 11... Done. (Resampled size: 77)
  Training SVM... Done (0.06s) | Acc: 0.5751
  Training DT... Done (0.06s) | Acc: 0.6597
  Training KNN... D

In [5]:
res_df = pd.DataFrame(results_all)
print("\nSummary Accuracy Table (%):")
display(res_df.pivot(index="Split", columns="Model", values="Accuracy").mul(100).round(2))


Summary Accuracy Table (%):


Model,DT,Hybrid DT-SVM,Hybrid KNN-SVM,KNN,SVM,XGBoost
Split,,,,,,
30-70,52.10,87.43,23.15,33.19,47.39,54.79
50-50,65.97,65.93,73.99,55.80,57.51,80.07
60-40,73.90,73.90,55.89,53.72,67.00,78.94
70-30,77.95,77.95,54.01,51.25,47.86,92.48
